# Transformer-based Image-captioning Model 

These are the modules we will need. 

In [1]:
!rm -rf /kaggle/working/*

In [1]:
%pip install tensordict

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 465.5/465.5 kB 12.2 MB/s eta 0:00:0000:01
Note: you may need to restart the kernel to use updated packages.


In [2]:
import torch
from torch import nn, optim, tensor, Tensor
from torch.utils.data import DataLoader
import tensordict as td 

import nltk, gensim.downloader as api
from gensim.models.keyedvectors import KeyedVectors

import json
from random import shuffle

In [3]:
BASE_PATH = '/kaggle/input/datasets/kanayevarma/coco-tensors-memmap' # where are the tensors
SAVE_TO_PATH = '/kaggle/working' # where to save results
DATASET_PATH = '/kaggle/input/datasets/kanayevarma/captions' # where are the captions

## Step 1: Pre-processing

In [11]:
class Vocab:
    # in case needed: vocab size is 22024
    def __init__(self, tokens : set[str]): 
        self.glove : KeyedVectors = api.load('glove-twitter-25') # type:ignore 
    
        # limit vocabulary 
        self.vocab_size = len(tokens) + 4 
        self.embedding = torch.randn(self.vocab_size, 25)
        self.embedding[0] = torch.from_numpy(self.glove.get_vector('unk').copy())
        self.word2idx = {
            'unk': 0, 
            '<start>' : 1, 
            '<stop>' : 2, 
            '<pad>' : 3
        }
        self.idx2word = ['unk', '<start>', '<stop>', '<pad>']

        valid_tokens = set()
        for token in tokens: 
            if self.glove.has_index_for(token): 
                self.word2idx[token] = len(self.idx2word) 
                self.idx2word.append(token)
                self.embedding[self.word2idx[token]] = torch.from_numpy(self.glove.get_vector(token).copy())
                valid_tokens.add(token)
        
        self.tokens = valid_tokens
        self.vocab_size = len(valid_tokens) + 4
        self.embedding = self.embedding[:self.vocab_size]

    @staticmethod
    def from_file(dataset_path): 
        tokens : set[str] = set()
        with open(dataset_path + '/captions_train2017.json') as f: 
            data = json.load(f)
            for item in data['annotations']: 
                caption : str = item['caption']
                for token in nltk.word_tokenize(caption.lower()): tokens.add(token)

        return Vocab(tokens) 

In [ ]:
vocab = Vocab.from_file(DATASET_PATH)

[==================================================] 100.0% 104.8/104.8MB downloaded


Now we can generate the training and validation datasets.

In [4]:
train_dataset_paths = [BASE_PATH + f'/{i}' for i in range(9)]
train_dataloaders = [
    DataLoader(
        td.load_memmap(path),
        batch_size=64,
        shuffle=True,
        num_workers=2, 
        collate_fn=lambda x : x
    ) for path in train_dataset_paths
]

In [5]:
val_dataset_path = BASE_PATH + '/9'
val_dataloader = DataLoader(
    td.load_memmap(val_dataset_path),
    batch_size=64,
    shuffle=False,
    num_workers=2, 
    collate_fn=lambda x : x
) 

## Step 2: Preparing the model

We start with the function to generate positional encodings for the transformer's encoder and decoder. 

In [6]:
def positional_encoding(seq_len : int, embed_dim : int, N_max : int = 10000) -> Tensor:
    '''Create positional encoding of size (seq_len, embed_dim)'''
    pos = torch.arange(seq_len).unsqueeze(1).expand(-1, embed_dim)
    idx = torch.arange(embed_dim).unsqueeze(1).expand(-1, seq_len).transpose(0, 1) 
    exp = N_max ** (idx.where(idx % 2 == 0, idx - 1) / embed_dim)
    return torch.where(idx % 2 == 0, torch.sin(pos/exp), torch.cos(pos/exp))

Next, we create the `Encoder` and `Decoder` modules for use later.

In [7]:
class Encoder(nn.Module):
    def __init__(self): 
        super().__init__()
        
        # 3 x (N, 14x14, 512) -> (N, 14x14, 512)
        self.attn = nn.MultiheadAttention(
            embed_dim=512, 
            num_heads=8,
            dropout=0.1, 
            batch_first=True,
        )
        # (N, 14x14, 512) -> (N, 14x14, 2048) -> (N, 14x14, 512)
        self.ffn = nn.Sequential(
            nn.Linear(512, 2048), 
            nn.ReLU(), 
            nn.Dropout(0.1),
            nn.Linear(2048, 512)

        )
        
        # (N, 14x14, 512) -> (N, 14x14, 512)
        self.lnorm1 = nn.LayerNorm(normalized_shape=512)
        self.lnorm2 = nn.LayerNorm(normalized_shape=512)
            
    def forward(self, x : Tensor): 
        
        # (N, 14x14, 512) throughout
        x_ = self.lnorm1(x)
        x = x + self.attn(x_, x_, x_, need_weights=False)[0]

        x = x + self.ffn(self.lnorm2(x))

        return x

In [8]:
class Decoder(nn.Module):
    def __init__(self, dev): 
        super().__init__()
        self.dev = dev 

        self.attn_1 = nn.MultiheadAttention(
            embed_dim=512, 
            num_heads=8, 
            dropout=0.1, 
            batch_first=True, 
        )
        self.attn_2 = nn.MultiheadAttention(
            embed_dim=512, 
            num_heads=8, 
            dropout=0.1, 
            batch_first=True, 
        )

        self.ffn = nn.Sequential(
            nn.Linear(512, 2048), 
            nn.ReLU(), 
            nn.Dropout(0.1), 
            nn.Linear(2048, 512)
        )

        self.lnorm1 = nn.LayerNorm(normalized_shape=512)
        self.lnorm2 = nn.LayerNorm(normalized_shape=512)
        self.lnorm3 = nn.LayerNorm(normalized_shape=512)
                
    def forward(self, x : Tensor, e : Tensor, mask : Tensor | None): 

        x_ = self.lnorm1(x)
        x = x + self.attn_1(x_, x_, x_, need_weights=False, 
            is_causal=True, 
            attn_mask=nn.Transformer.generate_square_subsequent_mask(x.shape[1]).float().to(self.dev),
            key_padding_mask=mask)[0]
        
        x = x + self.attn_2(self.lnorm2(x), e, e, need_weights=False)[0]

        x = x + self.ffn(self.lnorm3(x))

        return x 

Now we can assemble these into the final transformer

In [9]:
class CaptionModel(nn.Module): 
    def __init__(self, dev): 
        super().__init__()
        self.dev = dev 

        self.proj = nn.Sequential(
            nn.Linear(512, 512), 
            nn.ReLU(), 
            nn.Linear(512, 512), 
            nn.LayerNorm(512)
        )

        self.encoders = nn.ModuleList([Encoder() for _ in range(3)])

        self.embedding = nn.Embedding.from_pretrained(vocab.embedding, freeze=False)
        self.linear = nn.Linear(25, 512)
        self.decoders = nn.ModuleList([Decoder(self.dev) for _ in range(3)])

        self.ffn = nn.Linear(512, vocab.vocab_size)

        self.register_buffer(
            'pos_enc_x', positional_encoding(14*14, 512).unsqueeze(0)
        )
        self.register_buffer(
            'pos_enc_y',
            positional_encoding(20, 512)
        )

        self.encoder_final_norm = nn.LayerNorm(normalized_shape=512)
        self.decoder_final_norm = nn.LayerNorm(normalized_shape=512)
    
    def forward(self, x : Tensor, *args, scheduled_sampling : bool = False ):
        x = self.proj(x)
        x = x + self.pos_enc_x # type:ignore 
        for encoder in self.encoders: x = encoder(x) 
        x = self.encoder_final_norm(x)

        if scheduled_sampling: 
            if not args: raise SyntaxError('Need args for scheduled sampling')
            y : Tensor = args[0] # (N, L)
            mask : Tensor = args[1].float() # (N, L)
            N, L = y.shape 

            decoder_input = y[:, :1] # (N, 1)
            logits_list = [] # (N, 1, V)

            for t in range(1, L+1): 
                y2 = self.embedding(decoder_input) * (512 ** 0.5)
                y2 = self.linear(y2) # (N, t, 512)
                y2 = y2 + self.pos_enc_y[:t].unsqueeze(0) # type:ignore 
                
                for decoder in self.decoders: 
                    y2 = decoder(y2, x, mask[:, :t])
                
                y2 = self.decoder_final_norm(y2)
                logits : Tensor= self.ffn(y2) # (N, t, V)
                logits_list.append(logits[:, -1, :]) # (N, V)
                
                if t == L: break 
                if torch.rand(1).item() < 0.1: 
                    decoder_input = torch.cat([decoder_input,
                                               logits[:, -1, :].argmax(dim=-1, keepdim=True)], 
                                               dim=1) # (N, 1)
                else: 
                    decoder_input = torch.cat([decoder_input, 
                                               y[:, t:t+1]],
                                               dim=1)
                    
            return torch.stack(logits_list, dim=1) 

        if args: 
            '''model is training'''

            y : Tensor = args[0] # (N, L)
            y = self.embedding(y) * (512 ** 0.5) # (N, L, 25)
            y = self.linear(y) # (N, L, 512)

            y = y + self.pos_enc_y.unsqueeze(0) # type:ignore
            mask : Tensor = args[1].float()
            for decoder in self.decoders: 
                y = decoder(y, x, mask)
            
            y = self.decoder_final_norm(y)
            y = self.ffn(y) # (N, L, V)
            return y
        else: 
            '''model is evaluating'''

            outputs = [tensor([1], device=torch.device(self.dev))] # <start>
            pos = self.pos_enc_y

            while len(outputs) <= 20: 
                y = torch.cat(outputs).unsqueeze(0) # (1, L)
                y = self.embedding(y) * (512 ** 0.5) # (1, L, 25)
                y = self.linear(y) # (1, L, 512)
                
                y = y + pos[:y.shape[1]] # type:ignore 
                # (1, L, 512) 

                for decoder in self.decoders:
                    y = decoder(y, x, None)
                
                y = self.decoder_final_norm(y)
                # (1, L, 512) -> (1, L, V)
                y = self.ffn(y)

                # (1, L, V) -> (1, L) -> (1) [appended]
                outputs.append(torch.argmax(y, dim=2)[0, -1].unsqueeze(0)) 
                
                if outputs[-1].item() == 2: # <stop>
                    break  
            
            return torch.cat(outputs)

## Step 3: Training

### Note 
This notebook was recovered from when scheduled sampling was implemented after 3 epochs of training. As such, data for model training from epochs 1 to 3 has unfortunately been lost. The model from that point was loaded into a new Kaggle Notebook to continue training with scheduled sampling. 

The reason for the split was GPU time constraints on Kaggle requiring me to use 2 separate training sessions for the model. 

We can now initialise the model

In [13]:
model = CaptionModel('cuda')
model.to('cuda')
model.load_state_dict(
    torch.load('/kaggle/working/model3.pt', weights_only=True)
)
optimiser = optim.AdamW(model.parameters(), lr=1, weight_decay=0.01, betas=(0.9, 0.98))
optimiser.load_state_dict(
    torch.load('/kaggle/working/optim3.pt', weights_only=True)
)
loss_fn = nn.CrossEntropyLoss(ignore_index=3, label_smoothing=0.1)
epochs = 10

In [ ]:
def get_lr(step): 
    step = max(1, step)
    step += 8437 # account for jump from first 3 epochs 
    warmup = 4000
    d_model = 512 
    return (512 ** -0.5) * min(step ** -0.5, step * warmup ** -1.5)

scheduler = optim.lr_scheduler.LambdaLR(optimiser, get_lr)

These helper functions will be useful later on

In [21]:
def go_through(dataloader, t = 100, training=True, scheduled_sampling=0.0): 
    losses = []
    for batch in dataloader: 
        if len(losses) == t: 
            print("Loss:", sum(losses) / t)
            losses.clear()
        
        image_tensors = batch['images'].float().to('cuda')
        input_captions = batch['input_captions'].to('cuda')
        output_captions = batch['output_captions'].to('cuda')
    
        
        mask = input_captions == 3

        preds : Tensor = model(image_tensors, input_captions, mask, scheduled_sampling=scheduled_sampling)
        loss : Tensor = loss_fn(
            preds.flatten(0, 1), 
            output_captions.flatten(0, 1)
        )
        if training: 
            scheduler.step()
            optimiser.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimiser.step()

        losses.append(loss.item())        

    print("Loss:", sum(losses) / len(losses))

In [22]:
def train_one_epoch(e, scheduled_sampling):
    print("Training epoch:", e)
    model.train()
    shuffle(train_dataloaders)
    for train_dataloader in train_dataloaders:
        go_through(train_dataloader, training=True, scheduled_sampling=scheduled_sampling)
    print("Finished training epoch", e)
    print("Calculating validation losses...")
    model.eval()
    with torch.no_grad(): 
        go_through(val_dataloader, t=500, training=False)
    print("Completed epoch", e)


In [17]:
for e in range(4, epochs+1): 
    s = 0
    if e > 7: s = 0.3
    elif e > 5: s = 0.2
    elif e > 3: s = 0.1
    train_one_epoch(e, s)
    torch.save(model.state_dict(), SAVE_TO_PATH + f'/model{e}.pt')
    torch.save(optimiser.state_dict(), SAVE_TO_PATH + f'/optim{e}.pt')

Training epoch: 4


/tmp/ipykernel_55/938413420.py:21: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scheduler.step()


Loss: 3.846205589771271
Loss: 3.735910198688507
Loss: 3.7265885138511656
Loss: 3.6518887006319485
Loss: 3.7384136748313903
Loss: 3.675720055103302
Loss: 3.653602616786957
Loss: 3.648252065365131
Loss: 3.7878008031845094
Loss: 3.6677604126930237
Loss: 3.683380801677704
Loss: 3.6845205013568583
Loss: 3.7372905158996583
Loss: 3.662117528915405
Loss: 3.6148682904243468
Loss: 3.564348550943228
Loss: 3.8681375050544737
Loss: 3.7604790472984315
Loss: 3.7026986336708068
Loss: 3.670673572100126
Loss: 3.797739775180817
Loss: 3.6988096261024475
Loss: 3.6747341203689574
Loss: 3.610827849461482
Loss: 3.65892098903656
Loss: 3.576824984550476
Loss: 3.549753477573395
Loss: 3.611144395974966
Loss: 3.7126185941696166
Loss: 3.6720904731750488
Loss: 3.6499372220039366
Loss: 3.597004541983971
Loss: 3.877359685897827
Loss: 3.7862665367126467
Loss: 3.717544503211975
Loss: 3.7156989574432373
Finished training epoch 4
Calculating validation losses...
Loss: 3.6857888302483115
Completed epoch 4
Training epoch: 5

KeyboardInterrupt: 

In [18]:
torch.cuda.empty_cache()

The decision to use scheduled sampling was interrupted at this point out of concern for GPU time constraints. I decided to continue training the model from epoch 4 onwards without scheduled sampling. 

In [19]:
model = CaptionModel('cuda')
model.to('cuda')
model.load_state_dict(
    torch.load('/kaggle/working/model4.pt', weights_only=True)
)
optimiser = optim.AdamW(model.parameters(), lr=1, weight_decay=0.01, betas=(0.9, 0.98))
optimiser.load_state_dict(
    torch.load('/kaggle/working/optim4.pt', weights_only=True)
)
loss_fn = nn.CrossEntropyLoss(ignore_index=3, label_smoothing=0.1)
epochs = 10

In [ ]:
def get_lr(step): 
    step = max(1, step)
    step += 8437 # account for jump in training
    warmup = 4000
    d_model = 512 
    return (512 ** -0.5) * min(step ** -0.5, step * warmup ** -1.5)

scheduler = optim.lr_scheduler.LambdaLR(optimiser, get_lr)

In [24]:
for e in range(5, epochs+5): 
    train_one_epoch(e, 0.0)
    torch.save(model.state_dict(), SAVE_TO_PATH + f'/model{e}.pt')
    torch.save(optimiser.state_dict(), SAVE_TO_PATH + f'/optim{e}.pt')

Training epoch: 5
Loss: 3.4224666142463684
Loss: 3.4346292495727537
Loss: 3.429142174720764
Loss: 3.4346492290496826
Loss: 3.414948489665985
Loss: 3.34638329744339
Loss: 3.3144220066070558
Loss: 3.351337579580454
Loss: 3.5522262811660767
Loss: 3.4497511529922487
Loss: 3.436345021724701
Loss: 3.397885927787194
Loss: 3.5435207009315492
Loss: 3.482697353363037
Loss: 3.454720096588135
Loss: 3.5020965796250563
Loss: 3.5834197878837584
Loss: 3.513898251056671
Loss: 3.4549888586997985
Loss: 3.4670335329495945
Loss: 3.5651700115203857
Loss: 3.5196601009368895
Loss: 3.4891399216651915
Loss: 3.4420766830444336
Loss: 3.4967965984344485
Loss: 3.4444223141670225
Loss: 3.403963918685913
Loss: 3.4184458255767822
Loss: 3.480308356285095
Loss: 3.4116887497901915
Loss: 3.4004875779151917
Loss: 3.4026258908785305
Loss: 3.5870088744163513
Loss: 3.491506590843201
Loss: 3.4562901425361634
Loss: 3.4654225752903867
Finished training epoch 5
Calculating validation losses...
Loss: 3.7761209209125264
Completed e